# Intelligent Highway Lane Occupancy & Hogging Detection - Colab

This notebook is a thin wrapper around the `lane-analytics` CLI. The Python package is
not converted into notebook code.

Before running:

1. Select **Runtime > Change runtime type**. A GPU is optional; CPU works with `frame_stride`.
2. Put the project zip in Google Drive, or clone your repository into `/content`.
3. Put a highway clip in Google Drive, upload it, or record a public toll-road CCTV stream.
4. Run the cells from top to bottom.
5. Lane polygons must be calibrated for **your** camera. Use the preview cell before analysis.

The output is a rule-based **indication**, not a legal determination.


In [ ]:
import torch

print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    print("GPU not available; CPU inference still works (use frame_stride 2-3).")


## 1. Get the project

Option A (recommended): upload `highway-lane-occupancy.zip` to
`/content/drive/MyDrive/`.

Option B: clone your repository:

```python
!git clone https://github.com/<your-user>/highway-lane-occupancy.git /content/highway-lane-occupancy
```


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
import zipfile

PROJECT_DIR = "/content/highway-lane-occupancy"
ZIP_PATH = "/content/drive/MyDrive/highway-lane-occupancy.zip"

if not os.path.isdir(PROJECT_DIR):
    if os.path.exists(ZIP_PATH):
        with zipfile.ZipFile(ZIP_PATH) as archive:
            archive.extractall("/content")
    else:
        raise FileNotFoundError(
            f"Upload the project zip to {ZIP_PATH}, or clone your repo into {PROJECT_DIR}."
        )

%cd /content/highway-lane-occupancy
!ls


## 2. Install dependencies

Colab already provides CUDA-enabled PyTorch. Install Ultralytics and the project without
replacing the existing torch build.


In [ ]:
!pip install -q ultralytics
!pip install -q -e . --no-deps
!lane-analytics doctor


## 3. Select the input video

Recommended: upload the clip to Google Drive and set `VIDEO_PATH`.


In [ ]:
import os

VIDEO_PATH = "/content/drive/MyDrive/traffic-sample.mp4"  # <-- edit

# Alternative: upload a video from your computer.
# from google.colab import files
# uploaded = files.upload()
# VIDEO_PATH = "/content/" + next(iter(uploaded))

assert os.path.exists(VIDEO_PATH), f"Video not found: {VIDEO_PATH}"
print("Using:", VIDEO_PATH)


## 3a. Optional: record a public toll-road CCTV stream

The Semarang-Solo toll-road cameras publish live HLS playlists. They are live 24/7
streams, not downloadable videos. `record_stream` records 30-second chunks, reconnects
when the playlist rotates, and remuxes the result.

Use this responsibly: only the currently broadcasting window is available, and repeated
aggressive requests should be avoided.


In [ ]:
from lane_analytics.recorder import record_stream

CCTV_CAMERA = "km486_500"   # "km437_800" or "km486_500"
RECORD_SECONDS = 300        # lower for a quick test

CCTV_STREAMS = {
    "km437_800": {
        "url": "https://jmlive.jasamarga.com/hls/15/f541b422-2339-4896-b329-ae65bcb3cef0/index.m3u8",
        "camera_id": "tmj_km437_800",
    },
    "km486_500": {
        "url": "https://jmlive.jasamarga.com/hls/15/c59b1716-ea0a-43ff-b9b0-d26bd863a428/index.m3u8",
        "camera_id": "tmj_km486_500",
    },
}

assert CCTV_CAMERA in CCTV_STREAMS
stream = CCTV_STREAMS[CCTV_CAMERA]
VIDEO_PATH = f"/content/cctv_{CCTV_CAMERA}.mp4"
CAMERA_ID = stream["camera_id"]

result = record_stream(
    stream["url"],
    VIDEO_PATH,
    duration=RECORD_SECONDS,
    chunk_seconds=30,
)
print(f"Requested {result.requested_seconds:.0f}s -> recorded {result.actual_seconds:.1f}s "
      f"({result.chunk_count} chunk(s), {result.attempts} attempt(s))")
print("Complete:", result.complete)
print("Recorded:", VIDEO_PATH)
print("camera_id:", CAMERA_ID)


## 4. Configure lanes, roles, and hogging rules

Lane coordinates are **normalized** (`0.0 - 1.0`): `x / frame_width`, `y / frame_height`.
Edit the polygons below to match the camera. A good workflow:

1. Read one frame and note the lane corners in pixels.
2. Divide by width/height.
3. Keep a small gap at painted dividers.
4. Set `role` to `normal` or `overtaking` (or another label used by rules).

The default example assumes right-hand traffic with the rightmost lane as the overtaking
lane. Adapt it to the road.


In [ ]:
import yaml

CONFIG_PATH = "config/config.yaml"
with open(CONFIG_PATH, encoding="utf-8") as handle:
    config = yaml.safe_load(handle)

config["video"]["path"] = VIDEO_PATH
config["video"]["frame_stride"] = 2          # 1 for GPU, 2-3 for CPU
config["model"]["device"] = "0" if __import__("torch").cuda.is_available() else "cpu"
config["model"]["imgsz"] = 640
config["camera_id"] = globals().get("CAMERA_ID", config.get("camera_id", "camera_01"))
config["base_timestamp"] = ""                # "" = run start time

# Edit these polygons for your camera. Coordinates are normalized (0..1).
config["lanes"] = [
    {
        "name": "lane_left",
        "index": 1,
        "direction": "northbound",
        "role": "normal",
        "polygon": [[0.08, 0.10], [0.34, 0.10], [0.34, 0.96], [0.08, 0.96]],
    },
    {
        "name": "lane_center",
        "index": 2,
        "direction": "northbound",
        "role": "normal",
        "polygon": [[0.345, 0.10], [0.60, 0.10], [0.60, 0.96], [0.345, 0.96]],
    },
    {
        "name": "lane_right",
        "index": 3,
        "direction": "northbound",
        "role": "overtaking",
        "polygon": [[0.605, 0.10], [0.88, 0.10], [0.88, 0.96], [0.605, 0.96]],
    },
]

config["occupancy"]["anchor"] = "bottom_center"
config["occupancy"]["min_lane_frames"] = 2
config["occupancy"]["min_confidence"] = 0.30

config["hogging"]["enabled"] = True
config["hogging"]["retrigger_after_seconds"] = 0
config["hogging"]["rules"] = [
    {
        "name": "heavy_vehicle_in_overtaking_lane",
        "lane_roles": ["overtaking"],
        "classes": ["truck", "bus"],
        "min_dwell_seconds": 3.0,
        "severity": "high",
        "description": "Truk/bus menggunakan lajur overtaking melebihi batas waktu.",
    },
    {
        "name": "prolonged_overtaking_lane",
        "lane_roles": ["overtaking"],
        "classes": ["car"],
        "min_dwell_seconds": 12.0,
        "severity": "medium",
        "description": "Mobil bertahan di lajur overtaking melebihi batas waktu.",
    },
]

with open(CONFIG_PATH, "w", encoding="utf-8") as handle:
    yaml.safe_dump(config, handle, sort_keys=False)

print("Updated", CONFIG_PATH)
print("device:", config["model"]["device"], "| stride:", config["video"]["frame_stride"])


### 4a. Preview the lane polygons

Open the image and check that each polygon covers exactly one lane. If a vehicle's
bottom-centre anchor lands outside all polygons, the label becomes `lane n/a`.


In [ ]:
!lane-analytics preview-lanes --config config/config.yaml --frame 0 --out output/colab_lane_preview.jpg

from IPython.display import Image, display
display(Image(filename="output/colab_lane_preview.jpg"))


## 5. Run YOLO + ByteTrack

For a quick smoke test, add `--max-frames 300` or `--no-video`.


In [ ]:
import subprocess

command = [
    "lane-analytics", "analyze",
    "--config", "config/config.yaml",
    "--device", config["model"]["device"],
    "--frame-stride", str(config["video"]["frame_stride"]),
]
print(" ".join(command))
subprocess.run(command, check=True)


## 6. Play the annotated result

Browsers do not reliably play OpenCV's `mp4v` codec. The file is re-encoded to H.264 for
inline playback. For long clips, use `embed=False` or download the file.


In [ ]:
from pathlib import Path
import subprocess

videos = sorted(
    Path("output/videos").glob("*_annotated.mp4"),
    key=lambda path: path.stat().st_mtime,
)
assert videos, "No annotated video found - run the analysis cell first."
video = videos[-1]
browser_video = video.with_name(video.stem + "_h264.mp4")

subprocess.run(
    [
        "ffmpeg", "-y", "-loglevel", "error", "-i", str(video),
        "-vcodec", "libx264", "-pix_fmt", "yuv420p", "-movflags", "+faststart",
        str(browser_video),
    ],
    check=True,
)

from IPython.display import Video
print(browser_video)
Video(str(browser_video), embed=True, width=960)


## 7. Review the lane occupancy and hogging report


In [ ]:
from pathlib import Path
from IPython.display import Markdown, display
import pandas as pd

reports = sorted(
    Path("output/reports").glob("*/summary.md"),
    key=lambda path: path.stat().st_mtime,
)
assert reports, "No report found - run the analysis cell first."
report_dir = reports[-1].parent
display(Markdown(reports[-1].read_text(encoding="utf-8")))

for name in ("lane_usage.csv", "hogging_events.csv", "lane_changes.csv"):
    path = report_dir / name
    if path.exists():
        print(f"\n--- {name} ---")
        display(pd.read_csv(path).head(20))


## 8. Save results to Google Drive

Colab deletes `/content` when the runtime ends. Copy `output/` to Drive before closing.


In [ ]:
import shutil
from datetime import datetime
from pathlib import Path

destination = (
    Path("/content/drive/MyDrive/lane-occupancy-output")
    / datetime.now().strftime("%Y%m%d_%H%M%S")
)
shutil.copytree("output", destination, dirs_exist_ok=True)
print("Saved to", destination)


## Notes

- The offline synthetic demo also works on Colab (CPU is enough):

  ```python
  !lane-analytics demo --output-dir output/demo
  ```

- CPU inference: set `video.frame_stride` to 2-3 and `model.imgsz` to 480.
- Video decoding and OpenCV annotation still use CPU; `--no-video` maximizes data throughput.
- A track ID can switch after a long occlusion; review `lane_stints.csv` before drawing
  conclusions about a single vehicle.
- Lane-hogging rules are configurable heuristics. Review thresholds against the actual
  road rules before using the output for anything beyond research.
- Use footage you have the right to process. Ultralytics YOLO has its own license terms.
